In [36]:
from langchain_community.retrievers import WikipediaRetriever
import wikipedia

In [37]:
retriever=WikipediaRetriever(top_k_results=2,lang="en")

In [38]:
query = "Tell me about Einstien"

docs = retriever.invoke(query)

print(len(docs))
print(docs[0].page_content)

2
Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory. His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world's most famous equation". He received the 1921 Nobel Prize in Physics for "his services to theoretical physics, and especially for his discovery of the law of the photoelectric effect".
Born as a subject to the Kingdom of Württemberg, part of the German Empire, Einstein moved to Switzerland in 1895, forsaking his citizenship the following year. In 1896, at the age of seventeen, he enrolled in the mathematics and physics teaching diploma program at the Swiss federal polytechnic school in Zurich, graduating in 1900. He acquired Swiss citizenship a year later, which he kept for the rest of his life, and afterwards secured a permanent position at the Swiss Patent Office in Bern. In 

#### VectorStoreRetriever


In [39]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

In [40]:
# Step 1: Your source documents

documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [41]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [42]:
vectorstore=Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"

)

In [43]:
retriever=vectorstore.as_retriever(search_kwargs={"k":2})

In [44]:
query="What is Chroma used for?"
results=retriever.invoke(query)

In [45]:
for i,doc in enumerate(results):
    print(doc.page_content)

Chroma is a vector database optimized for LLM-based search.
Chroma is a vector database optimized for LLM-based search.


#### MMR(Maximal Marginal Relevance)

In [46]:
# Simple documents

docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM-based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [47]:
from langchain_community.vectorstores import FAISS

embedding_model=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vectorstore=FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [48]:
retriever=vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={'k':2,"lambda_mult":1}
)

In [49]:
query="What is Langchain?"
result=retriever.invoke(query)

In [50]:
for i,doc in enumerate(result):
    print(f"\n...Result{i+1}...")
    print(doc.page_content)



...Result1...
LangChain is used to build LLM-based applications.

...Result2...
LangChain makes it easy to work with LLMs.


#### MultiQuery Retriever


In [51]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAI
from langchain_classic.retrievers import MultiQueryRetriever

In [52]:
# Relevant health & wellness documents

all_docs = [
    Document(
        page_content="Regular walking boosts heart health and can reduce symptoms of depression.",
        metadata={"source": "H1"}
    ),
    Document(
        page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.",
        metadata={"source": "H2"}
    ),
    Document(
        page_content="Omega-3 is essential for cellular repair and emotional regulation.",
        metadata={"source": "H3"}
    ),
    Document(
        page_content="Meditation and deep breathing lower cortisol and improve mental clarity.",
        metadata={"source": "H4"}
    ),
    Document(
        page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.",
        metadata={"source": "H5"}
    ),
    Document(
        page_content="The solar energy system in modern homes helps balance electricity demand.",
        metadata={"source": "I1"}
    ),
    Document(
        page_content="Python balances readability with power, making it a popular system design language.",
        metadata={"source": "I2"}
    ),
    Document(
        page_content="Photosynthesis enables plants to produce energy by converting sunlight.",
        metadata={"source": "I3"}
    ),
    Document(
        page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.",
        metadata={"source": "I4"}
    ),
    Document(
        page_content="Black holes bend space-time and store immense gravitational energy.",
        metadata={"source": "I5"}
    ),
]

In [53]:
embedding_model=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vectorstore=FAISS.from_documents(documents=all_docs,embedding=embedding_model)

In [54]:
similarity_retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={'k':5})
multiquery_retriever=MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k':5}),
    llm=GoogleGenerativeAI(model='gemini-3.8-flash')
)

In [55]:
query="How to improve energy levels and maintain balance?"

In [56]:
similarity_results=similarity_retriever.invoke(query)
multiquery_results=multiquery_retriever.invoke(query)

In [57]:
for i,docs in enumerate(similarity_results):
    print(f"\n...Result{i+1}...")
    print(doc.page_content)


...Result1...
LangChain makes it easy to work with LLMs.

...Result2...
LangChain makes it easy to work with LLMs.

...Result3...
LangChain makes it easy to work with LLMs.

...Result4...
LangChain makes it easy to work with LLMs.

...Result5...
LangChain makes it easy to work with LLMs.


In [58]:
for i,docs in enumerate(multiquery_results):
    print(f"\n...Result{i+1}...")
    print(doc.page_content)


...Result1...
LangChain makes it easy to work with LLMs.

...Result2...
LangChain makes it easy to work with LLMs.

...Result3...
LangChain makes it easy to work with LLMs.

...Result4...
LangChain makes it easy to work with LLMs.

...Result5...
LangChain makes it easy to work with LLMs.


#### Context Compression Retriever

In [63]:
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_community.vectorstores import FAISS

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


# ==========================================
# 1. Load environment variables
# ==========================================

load_dotenv()


# ==========================================
# 2. Create documents
# ==========================================

docs = [
    Document(
        page_content="""
        Regular physical activity provides many health benefits.
        Walking, running, and cycling can improve cardiovascular health.
        Regular exercise can strengthen the heart and improve blood circulation.
        Exercise may also help reduce stress, improve mood, and support better sleep.
        Adults should aim for regular physical activity throughout the week.
        """,
        metadata={"source": "health_1"}
    ),

    Document(
        page_content="""
        A healthy diet should contain vegetables, fruits, whole grains,
        proteins, and healthy fats. Leafy green vegetables provide vitamins
        and minerals. Drinking enough water is also important for maintaining
        normal body functions.
        """,
        metadata={"source": "health_2"}
    ),

    Document(
        page_content="""
        Solar panels convert sunlight into electrical energy.
        Solar energy can reduce dependence on fossil fuels.
        Modern photovoltaic systems are increasingly being installed
        on residential and commercial buildings.
        """,
        metadata={"source": "energy_1"}
    ),

    Document(
        page_content="""
        Python is a popular programming language used for machine learning,
        data science, web development, and automation. Its simple syntax
        makes it accessible to beginners.
        """,
        metadata={"source": "technology_1"}
    ),
]


# ==========================================
# 3. Create Gemini Embeddings
# ==========================================

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)


# ==========================================
# 4. Create FAISS Vector Store
# ==========================================

vectorstore = FAISS.from_documents(
    docs,
    embeddings
)


# ==========================================
# 5. Create Base Retriever
# ==========================================

base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)


# ==========================================
# 6. Create Gemini LLM
# ==========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)


# ==========================================
# 7. Create LLM Compressor
# ==========================================

compressor = LLMChainExtractor.from_llm(llm)


# ==========================================
# 8. Create Contextual Compression Retriever
# ==========================================

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)


# ==========================================
# 9. Query
# ==========================================

query = "What are the health benefits of regular physical exercise?"


# ==========================================
# 10. Normal Retrieval
# ==========================================

normal_docs = base_retriever.invoke(query)

print("\n" + "=" * 70)
print("NORMAL RETRIEVAL")
print("=" * 70)

for i, doc in enumerate(normal_docs, start=1):

    print(f"\nDocument {i}")
    print(f"Source: {doc.metadata['source']}")
    print("-" * 50)
    print(doc.page_content.strip())


# ==========================================
# 11. Contextual Compression Retrieval
# ==========================================

compressed_docs = compression_retriever.invoke(query)

print("\n" + "=" * 70)
print("COMPRESSED RETRIEVAL")
print("=" * 70)

for i, doc in enumerate(compressed_docs, start=1):

    print(f"\nDocument {i}")
    print(f"Source: {doc.metadata['source']}")
    print("-" * 50)
    print(doc.page_content.strip())


NORMAL RETRIEVAL

Document 1
Source: health_1
--------------------------------------------------
Regular physical activity provides many health benefits.
        Walking, running, and cycling can improve cardiovascular health.
        Regular exercise can strengthen the heart and improve blood circulation.
        Exercise may also help reduce stress, improve mood, and support better sleep.
        Adults should aim for regular physical activity throughout the week.

Document 2
Source: health_2
--------------------------------------------------
A healthy diet should contain vegetables, fruits, whole grains,
        proteins, and healthy fats. Leafy green vegetables provide vitamins
        and minerals. Drinking enough water is also important for maintaining
        normal body functions.

Document 3
Source: energy_1
--------------------------------------------------
Solar panels convert sunlight into electrical energy.
        Solar energy can reduce dependence on fossil fuels.
     